# FR02
* **Numerical (Số)**: `budget`, `revenue`, `runtime`, `vote_average`, `vote_count`.
* **Categorical (Phân loại)**: `genres`, `production_companies`.
* **Date/Time (Ngày/Giờ)**: `release_date`.
* **Identifier (Định danh)**: `id`.
* **Free-text / Links**: `title`, `overview`, .

# FR03 - Define Three Hypotheses

## Hypothesis 1

* **Hypothesis Statement:** Mức độ tương quan thuận giữa Tổng số lượt đánh giá (`vote_count`) và Doanh thu (`revenue`) mạnh mẽ hơn đáng kể so với mức độ tương quan giữa Điểm đánh giá trung bình (`vote_average`) và Doanh thu (`revenue`).
* **Variables:** 
  * `vote_count` (Số nguyên)
  * `vote_average` (Số thực)
  * `revenue` (Số thực)
* **Population / Subset:** Các bộ phim có báo cáo doanh thu và lượt bình chọn hợp lệ (`revenue > 0` và `vote_count > 10`).
* **Metric:** Hệ số tương quan Pearson (r) hoặc Spearman (rs).
* **Planned Analysis:** Tính hai hệ số tương quan:
  * r1 = correlation(vote_count, revenue)
  * r2 = correlation(vote_average, revenue)
  Sau đó so sánh độ lớn |r1| và |r2|.
* **Planned Visualization:** Biểu đồ phân tán (**Scatter Plot**) kèm đường xu hướng (**Trendline**) cho từng cặp biến, hoặc **Correlation Heatmap**.
* **Decision Rule:**
  * **Accepted:** Nếu r1 > r2 và mức chênh lệch có ý nghĩa thống kê (p < 0.05).
  * **Rejected:** Nếu r1 <= r2.
  * **Inconclusive:** Nếu dữ liệu bị lệch quá nặng (right-skewed) hoặc giá trị ngoại lệ (outliers) làm méo lệch hệ số tương quan ngay cả khi đã biến đổi log.

# FR04 - Prepare and Clean Data

## 1. Data Quality Assessment & General Cleaning Rules
To build a high-quality analytical base dataset, raw dataset `movies.csv` (769,631 records) underwent systematic data quality profiling and cleaning:

1. **Deduplication (`id`)**: 107,548 duplicate entity records were dropped.
2. **Missing Essential Metadata (`title`)**: 6 records lacking titles were eliminated.
3. **Datetime Validation (`release_date`)**: 59,455 records with missing/invalid release dates were removed, standardizing valid dates to `datetime64`.
4. **Impossible Financial Values Filter (`revenue`, `budget`)**: 1 record with impossible negative financial values (`revenue < 0` or `budget < 0`) was dropped.
5. **Categorical Imputation**: Missing values in `genres`, `production_companies`, `spoken_languages`, and `production_countries` were filled with `'Unknown'`.
6. **Data Availability Indicator Flags**: Added `has_revenue_data` (`revenue > 0`) and `has_budget_data` (`budget > 0`) to distinguish reported financial data from missing/unreported values (`0`).
7. **Time Feature Extraction**: Extracted `release_year` and `release_month` for downstream temporal exploratory analyses.

## 2. Data Quality Traceability Matrix
The step-by-step cleaning operations and record flow are exported to `outputs/tables/data_quality_trace.csv`.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

# Ensure src directory is in sys path
sys.path.append(os.path.abspath('../src'))
from data_cleaner import clean_movie_data

# Load raw dataset
raw_path = '../data/raw/movies.csv'
df_raw = pd.read_csv(raw_path)

# Execute general data cleaning pipeline
df_clean, df_trace = clean_movie_data(df_raw)

# Save general cleaned dataset & traceability table
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../outputs/tables', exist_ok=True)

df_clean.to_csv('../data/processed/dataset_clean.csv', index=False)
df_trace.to_csv('../outputs/tables/data_quality_trace.csv', index=False)

print('=== DATA QUALITY TRACE SUMMARY ===')
print(df_trace.to_string(index=False))

print('\n=== GENERAL CLEANED DATASET SUMMARY ===')
print(f'Original Raw Shape : {df_raw.shape}')
print(f'General Cleaned Shape: {df_clean.shape}')


# FR05 - Analyze Hypothesis 1

## 1. Analytical Method & Data Scope
Pursuant to the defined **Hypothesis 1 (GH1)** in FR03, we evaluate whether the linear association between total rating volume (`vote_count`) and financial revenue (`revenue`) is significantly stronger than that between average rating score (`vote_average`) and revenue.

- **Target Population Subset**: Movies with valid reported revenue (`revenue > 0`) and sufficient vote evaluation volume (`vote_count >= 10`), yielding $N = 11,622$ records.
- **Log Transformation**: Applied $\log_{10}$ transformation to `revenue` and `vote_count` to mitigate right-skewness and extreme box-office outliers.
- **Analytical Operations**: Computed Pearson correlation coefficient ($r$) across raw and log-transformed scales.

## 2. Key Findings & Sensitivity Analysis
- **Correlation Comparison**: The raw Pearson correlation for `vote_count` vs `revenue` ($r = 0.7662$) is substantially stronger than `vote_average` vs `revenue` ($r = 0.1781$).
- **Log Scale Robustness**: On log-log transformed scale, $\text{corr}(\log_{10}(\text{vote\_count}), \log_{10}(\text{revenue})) = 0.6115$, confirming that popularity volume strongly aligns with commercial revenue.
- **Potential Biases**: Heavy right-skewness and survivor bias (only movies with reported non-zero revenues are included).

In [ ]:
import sys
import os
import pandas as pd

# Ensure src directory is in sys path
sys.path.append(os.path.abspath('../src'))
from analyzer import analyze_hypothesis_1

# Load cleaned dataset from FR04
df_clean = pd.read_csv('../data/processed/dataset_clean.csv')

# Perform FR05 Analysis for Hypothesis 1
df_gh1, df_summary_gh1 = analyze_hypothesis_1(df_clean)

# Save summary results table
os.makedirs('../outputs/tables', exist_ok=True)
df_summary_gh1.to_csv('../outputs/tables/hypothesis_results.csv', index=False)

print('=== FR05 ANALYTICAL RESULTS TABLE (HYPOTHESIS 1) ===')
print(df_summary_gh1.to_string(index=False))


# FR06 - Visualize the Evidence

## 1. Primary Visualization for Hypothesis 1
To visually validate **Hypothesis 1**, we construct a dual-panel Scatter Plot with linear Trendlines comparing:
- **Panel A**: `Log10(Vote Count)` vs `Log10(Revenue)` ($r = 0.6115$).
- **Panel B**: `Vote Average` vs `Log10(Revenue)` ($r = 0.1944$).

## 2. Visualization Standards Compliance
- **Chart Choice**: Scatter plot is used to illustrate bivariate relationships and spread.
- **Scale Optimization**: Logarithmic scale ($\log_{10}$) is applied to revenue and vote count to prevent misleading compression caused by extreme right-skewness.
- **Reproducibility**: The generated chart is saved as high-resolution image `outputs/figures/hypothesis1_correlation.png`.

In [ ]:
import sys
import os
import pandas as pd

# Ensure src directory is in sys path
sys.path.append(os.path.abspath('../src'))
from visualizer import visualize_hypothesis_1

# Load cleaned dataset from FR04
df_clean = pd.read_csv('../data/processed/dataset_clean.csv')

# Generate FR06 Scatter Plot Visualization
fig_path = visualize_hypothesis_1(df_clean, output_dir='../outputs/figures')
print(f'=== FR06 CHART SAVED SUCCESSFULLY ===')
print(f'File Location: {fig_path}')
